# 02 — Baseline Models

**Research use only. Not for clinical decisions.**

Set `DATA_PATH` and `TARGET_COLUMN` before running.

In [ ]:
import os, sys
sys.path.insert(0, '../src')
DATA_PATH = os.environ.get('DATA_PATH', '')
TARGET_COLUMN = os.environ.get('TARGET_COLUMN', 'severe')
if not DATA_PATH:
    raise RuntimeError('Set DATA_PATH. No dataset is bundled.')

In [ ]:
from penux_ap.datasets import load_dataset
from penux_ap.labels import binarize_target
from penux_ap.preprocessing import build_preprocessor, infer_feature_types, make_train_test_split
from penux_ap.models import get_model_registry, train_model, predict_proba_safe
from penux_ap.evaluation import evaluate_binary_classifier

df = load_dataset(DATA_PATH)
y = binarize_target(df[TARGET_COLUMN]).dropna().astype(int)
X = df.drop(columns=[TARGET_COLUMN]).loc[y.index]
types = infer_feature_types(df.loc[y.index], TARGET_COLUMN)
preprocessor = build_preprocessor(types['numeric'], types['categorical'])
X_train, X_test, y_train, y_test = make_train_test_split(X, y)

In [ ]:
results = {}
for name in get_model_registry():
    try:
        model = train_model(name, X_train, y_train, preprocessor)
        proba = predict_proba_safe(model, X_test)
        results[name] = evaluate_binary_classifier(y_test.values, proba)
        print(f"{name}: AUROC={results[name]['auroc']:.3f} AUPRC={results[name]['auprc']:.3f}")
    except Exception as e:
        print(f"{name}: FAILED — {e}")